# A deterministic dummy agent loop

A complete, deterministic course companion.

## Learning objectives

- Represent a conversation as explicit role/content messages.
- Trace one Thought, Action, and Observation cycle.
- Separate a deterministic tool from the agent policy that calls it.

## Setup

The offline lesson uses only Python data structures and needs no credentials. The optional live cell uses `smolagents` and is guarded by `HF_TOKEN`.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
message_history = [
    {"role": "system", "content": "Answer weather questions by using the weather tool."},
    {"role": "user", "content": "What is the weather in Riga?"},
]

def deterministic_weather_tool(city: str) -> str:
    weather = {"Riga": {"condition": "clear", "temperature": 21}}
    if city not in weather:
        return f"No offline weather record for {city}."
    record = weather[city]
    return f"Weather in {city}: {record['condition']}, {record['temperature']}°C"

trace = []
thought = "I need current weather data, so I should call the weather tool."
trace.append({"stage": "Thought", "content": thought})

action = {"tool": "deterministic_weather_tool", "arguments": {"city": "Riga"}}
trace.append({"stage": "Action", "content": action})

weather_result = deterministic_weather_tool(**action["arguments"])
trace.append({"stage": "Observation", "content": weather_result})
message_history.append({"role": "tool", "content": weather_result})

final_answer = "Riga is clear at 21°C."
message_history.append({"role": "assistant", "content": final_answer})

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert [step["stage"] for step in trace] == ["Thought", "Action", "Observation"]
assert weather_result == "Weather in Riga: clear, 21°C"
assert final_answer == "Riga is clear at 21°C."
assert message_history[-1] == {"role": "assistant", "content": final_answer}

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The trace makes the control flow observable: reasoning selects a tool, the action supplies structured arguments, and the observation grounds the final response. A real model can choose actions dynamically, but the interface boundary remains the same.

## Credentials

No credential is used by the offline cells. The live example reads `HF_TOKEN` from the environment and never displays its value.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import CodeAgent
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token
            )
        live_agent = CodeAgent(tools=[], model=model, max_steps=2)
        live_answer = live_agent.run("In one sentence, explain the Thought-Action-Observation cycle.")
        print(live_answer)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ In one sentence, explain the Thought-Action-Observation cycle.                                                  │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
The Thought-Action-Observation cycle is a method used in problem-solving where we analyze the problem, take action 
based on our analysis, and observe the results. This process helps us identify the root cause of the problem and 
develop effective solutions.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

[Step 1: Duration 71.75 seconds| Input tokens: 2,038 | Output tokens: 50]

... (3 outputs left out)

[Step 2: Duration 91.92 seconds| Input tokens: 4,290 | Output tokens: 181]

Reached max steps.

[Step 3: Duration 26.71 seconds| Input tokens: 4,918 | Output tokens: 231]

The Thought-Action-Observation cycle is a method used in problem-solving where we analyze the problem, take action based on our analysis, and observe the results. This process helps us identify the root cause of the problem and develop effective solutions.


## Completion checklist

- [ ] Inspected the message history.
- [ ] Ran a Thought/Action/Observation cycle.
- [ ] Verified the grounded final answer.

## Attribution

Original companion example informed by the [Hugging Face Agents Course dummy-agent lesson](https://huggingface.co/learn/agents-course/unit1/dummy-agent-library) and the [smolagents tutorial](https://huggingface.co/learn/agents-course/unit1/tutorial). No official notebook text or full implementation is reproduced. Notebook code is released under the Apache-2.0 metadata license.